# 07 — Visualization and quality control

- `rocqipath.viz.view_pairs` — look at matched patch pairs;
- `rocqipath.viz.plot_selector_map` — grid maps of selected tissue cells;
- `rp.overlay_markers` — several IHC markers as colored masks on one base;
- `rp.compare` — publication figures of H&E, true IHC and predicted IHC;
- `rp.open_slide` — read matching fields from aligned slides for figures.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
PATCH_CASE = RESULTS_ROOT / "patches" / "patch_extraction" / "Sample_0001_CD8"
OVERLAY_CASES = DATA_ROOT / "overlay_patches"     # <case>/<marker>/<patch>.png
OUTPUT_DIR = RESULTS_ROOT / "figures"

RUN_OVERLAYS = False
RUN_COMPARE = False

In [ ]:
from rocqipath.viz import view_pairs

if PATCH_CASE.is_dir():
    view_pairs(str(PATCH_CASE), num_to_show=5)
else:
    print(f"No patch case at {PATCH_CASE} (run notebook 04).")

In [ ]:
from PIL import Image, ImageDraw

from rocqipath.viz import plot_selector_map

thumb = Image.new("RGB", (800, 600), (245, 245, 245))
ImageDraw.Draw(thumb).ellipse((80, 90, 710, 520), fill=(205, 145, 180))
grid_png = DEMO_ROOT / "visualization" / "grid_map_demo.png"
grid_png.parent.mkdir(parents=True, exist_ok=True)
plot_selector_map(thumb, {7, 8, 9, 10, 13, 14, 15, 16, 19, 20, 21, 22, 26, 27}, 6, 6,
                  output_path=str(grid_png), show=True)

## Marker overlays

Each case folder holds one subfolder per marker with identical filenames:

```text
overlay_patches/Sample_0001/
├── CD8/patch_000001.png
├── CD31/patch_000001.png
└── CAIX/patch_000001.png
```

In [ ]:
overlay_settings = dict(
    markers={
        "CD8": rp.MarkerProfile(color=(220, 30, 45)),
        "CD31": rp.MarkerProfile(color=(20, 135, 210)),
        "CAIX": rp.MarkerProfile(color=(250, 195, 25)),
    },
    combinations=[rp.OverlayCombo(base="CD31", overlays=["CD8", "CAIX"])],
    base_marker="CD31",
    base_render_mode="original",
    plot_mode="both",
    patches_per_case=10,
    dpi=300,
)

if RUN_OVERLAYS:
    overlays = rp.overlay_markers(OVERLAY_CASES, OUTPUT_DIR, **overlay_settings)
    print(overlays.summary)
else:
    print("Set RUN_OVERLAYS=True after organizing OVERLAY_CASES.")

## Publication comparison figures

In [ ]:
HE = DATA_ROOT / "wsi" / "Sample_0001_he.tif"
REAL_IHC = DATA_ROOT / "wsi" / "Sample_0001_cd8.tif"
PREDICTED_IHC = DATA_ROOT / "wsi" / "Sample_0001_cd8_pred.tif"

if RUN_COMPARE:
    figures = rp.compare([HE, REAL_IHC, PREDICTED_IHC], OUTPUT_DIR,
                         zooms=["20x", "10x"], random_rois=3, scale_bars=True, dpi=600)
    print([item.path.name for item in figures])
else:
    print("Set RUN_COMPARE=True after editing the three paths.")

**QC checklist** — inspect low-texture and edge regions; confirm structures
line up at the same coordinates; check mask specificity on weak stain, red
cells, pigment and folds; use 300 DPI for QC and 600 DPI (or PDF) for
publication.